# Final Model Selection Audit After GBM Leakage Correction

## 1. Objective

This notebook evaluates how the original T38 model selection changes after replacing the original GBM and Hybrid predictions with the corrected versions.

The analysis covers 14 energy consumption series and follows the original project evaluation framework.

## 2. Methodology

The audit will:

1. Inspect and validate the original T36 and T38 evaluation scripts.
2. Reconstruct the prediction dataset using the original Chronos, Random Forest, CatBoost and LightGBM outputs, together with the corrected GBM and Hybrid predictions.
3. Evaluate candidate models on common timestamps during the SEL period (October 16 – December 31, 2022).
4. Compare the updated model selection with the original T38 results.
5. Recalculate relevant evaluation metrics and document any changes.

## 3. Limitations

The corrected GBM and Hybrid are experimental fixed-horizon forecasting models.

The original Random Forest and CatBoost predictions have not been retrained using strict day-ahead features and may include same-hour sensor information. Therefore, this analysis is a partial leakage-correction audit, not a fully leakage-free comparison across all model families.

Model selection must use SEL only. The 2023 and 2025 test periods will be used for subsequent evaluation, not model selection.

In [ ]:

# STEP 2 - Upload project files

from google.colab import files
from pathlib import Path
import pandas as pd
import numpy as np
import zipfile

uploaded = files.upload()

print("\nUploaded files:")
for name in uploaded:
    print("-", name)


Saving Hybrid_Corrected_Predictions_10series (1).csv to Hybrid_Corrected_Predictions_10series (1) (1).csv
Saving GBM_Corrected_Predictions_10series (1).csv to GBM_Corrected_Predictions_10series (1) (1).csv
Saving 2026G3-main.zip to 2026G3-main (1).zip

Uploaded files:
- Hybrid_Corrected_Predictions_10series (1) (1).csv
- GBM_Corrected_Predictions_10series (1) (1).csv
- 2026G3-main (1).zip


In [ ]:

# STEP 3 - Extract original repository

zip_names = [
    name for name in uploaded
    if name.startswith("2026G3-main")
    and name.lower().endswith(".zip")
]

assert len(zip_names) == 1, (
    f"Expected one project ZIP, found: {zip_names}"
)

zip_file = Path("/content") / zip_names[0]

extract_dir = Path("/content/tbda_selection")
extract_dir.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_file) as z:
    z.extractall(extract_dir)

project_dir = extract_dir / "2026G3-main"

assert project_dir.exists(), "Project folder not found"

print("Repository loaded:", project_dir)

required_files = [
    "predictions_and_eval/T36_combine_predictions.py",
    "predictions_and_eval/TeamB_T38_final_evaluation.py",
    "predictions_and_eval/best_model_selection_final.csv"
]

for relative_path in required_files:
    path = project_dir / relative_path
    print(
        "FOUND" if path.exists() else "MISSING",
        relative_path
    )
    assert path.exists(), f"Missing: {relative_path}"

print("\nRepository verification PASSED")


Repository loaded: /content/tbda_selection/2026G3-main
FOUND predictions_and_eval/T36_combine_predictions.py
FOUND predictions_and_eval/TeamB_T38_final_evaluation.py
FOUND predictions_and_eval/best_model_selection_final.csv

Repository verification PASSED


In [ ]:

# STEP 4 - Load corrected GBM and Hybrid predictions

def find_uploaded(prefix):
    matches = [
        name for name in uploaded
        if name.startswith(prefix)
        and name.lower().endswith(".csv")
    ]
    assert len(matches) == 1, (
        f"Expected one {prefix} CSV, found: {matches}"
    )
    return Path("/content") / matches[0]

gbm_file = find_uploaded(
    "GBM_Corrected_Predictions_10series"
)

hybrid_file = find_uploaded(
    "Hybrid_Corrected_Predictions_10series"
)

gbm_new = pd.read_csv(gbm_file)
hybrid_new = pd.read_csv(hybrid_file)

required_columns = [
    "timestamp_utc",
    "zone",
    "load_type",
    "model",
    "split",
    "y_true",
    "y_pred"
]

for name, data in [
    ("GBM", gbm_new),
    ("Hybrid", hybrid_new)
]:
    assert data.columns.tolist() == required_columns

    data["timestamp_utc"] = pd.to_datetime(
        data["timestamp_utc"], utc=True
    )

    assert not data.duplicated(
        ["timestamp_utc", "zone", "load_type"]
    ).any()

    assert data[["y_true", "y_pred"]].notna().all().all()

    print(
        name,
        "| Rows:", len(data),
        "| Series:", data.groupby(
            ["zone", "load_type"]
        ).ngroups,
        "| Models:", data["model"].unique().tolist()
    )

assert gbm_new["model"].eq("gbm_leakfree").all()
assert hybrid_new["model"].eq("hybrid_leakfree").all()

print("\nCorrected prediction files PASSED")


GBM | Rows: 46007 | Series: 10 | Models: ['gbm_leakfree']
Hybrid | Rows: 46007 | Series: 10 | Models: ['hybrid_leakfree']

Corrected prediction files PASSED


In [ ]:

# STEP 5 - Inspect official T36 script

t36_path = (
    project_dir /
    "predictions_and_eval/T36_combine_predictions.py"
)

assert t36_path.exists(), "T36 script not found"

t36_source = t36_path.read_text(encoding="utf-8")
t36_lines = t36_source.splitlines()

print("T36 script:", t36_path.name)
print("Total lines:", len(t36_lines))

keywords = [
    "def collect",
    "rglob",
    "zipfile",
    "timestamp_utc",
    "drop_duplicates",
    "sys.argv",
    "argparse",
    "__main__"
]

for i, line in enumerate(t36_lines):
    if any(word in line for word in keywords):
        print(f"{i+1:03d}: {line}")


T36 script: T36_combine_predictions.py
Total lines: 101
007:   2. Keeps only files in the agreed format: timestamp_utc, zone, load_type, model, split, y_true, y_pred.
014:      y_true matches energy_wh_model at that UTC hour. If a model saved LOCAL time in `timestamp_utc`
019: import glob, io, os, sys, zipfile
022: NEED = ["timestamp_utc", "zone", "load_type", "model", "split", "y_true", "y_pred"]
041: def collect(folder):
052:             with zipfile.ZipFile(path) as z:
063: if __name__ == "__main__":
064:     folder, out = sys.argv[1], sys.argv[2]
070:     p["timestamp_utc"] = pd.to_datetime(p["timestamp_utc"])
072:     if len(sys.argv) > 3:                                   # timestamp check against the clean dataset
073:         c = pd.read_csv(os.path.join(sys.argv[3], "cedint_f1_hourly_clean.csv.gz"),
074:                         usecols=["timestamp_utc", "timestamp_local", "zone", "load_type", "energy_wh_model"],
075:                         parse_dates=["timestamp_utc", "times

In [ ]:

# STEP 6 - Inspect original T38 selection logic

t38_path = (
    project_dir /
    "predictions_and_eval/TeamB_T38_final_evaluation.py"
)

assert t38_path.exists(), "T38 script not found"

t38_lines = t38_path.read_text(
    encoding="utf-8"
).splitlines()

print("T38 selection logic:\n")

keywords = [
    "SEL",
    "groupby",
    "pool",
    "nunique",
    "cands",
    "best =",
    "best=",
    "common",
    "sys.argv",
    "argparse"
]

matching_lines = [
    i for i, line in enumerate(t38_lines)
    if any(word in line for word in keywords)
]

shown = set()

for i in matching_lines:
    for j in range(
        max(0, i - 2),
        min(len(t38_lines), i + 4)
    ):
        shown.add(j)

for j in sorted(shown):
    print(f"{j+1:03d}: {t38_lines[j]}")


T38 selection logic:

030: 
031: BENCH = {"naive_day", "naive_week", "seasonalnaive"}
032: SEL = ("2022-10-16 00:00", "2022-12-31 23:00")          # out-of-sample half of validation for every model
033: SPLITS = ["validation", "test_2023", "test_2025"]
034: 
035: 
077:     On = first hour above 50 % of that day's maximum; off = last such hour; for actual and forecast separately."""
078:     on_e, off_e = [], []
079:     for _, d in df.groupby("date"):
080:         if len(d) < 20 or d["y_true"].max() <= 0 or d["y_pred"].max() <= 0:
081:             continue
082:         a = d[d["y_true"] > 0.5 * d["y_true"].max()]["hour"]
089: 
090: def daily(df):
091:     d = (df.groupby("date").agg(y_true=("y_true", "sum"), y_pred=("y_pred", "sum"), h=("y_true", "size"))
092:          .reset_index())
093:     return d[d["h"] >= 20]
094: 
095: 
096: if __name__ == "__main__":
097:     pred_path, clean_dir, t30_path, out = sys.argv[1:5]
098:     os.makedirs(out, exist_ok=True)
099:     p = pd.read_csv(p

In [ ]:

# STEP 7 - Inspect T36 input and output handling

for i, line in enumerate(t36_lines, start=1):
    if 20 <= i <= 101:
        print(f"{i:03d}: {line}")


020: import pandas as pd
021: 
022: NEED = ["timestamp_utc", "zone", "load_type", "model", "split", "y_true", "y_pred"]
023: SPLITS = {"validation": ("2022-07-01", "2022-12-31 23:00"),
024:           "test_2023": ("2023-01-01", "2023-02-28 23:00"), "test_2025": ("2025-08-18", "2025-10-13 23:00")}
025: 
026: 
027: def read_any(src, name, opener):
028:     try:
029:         with opener() as fh:
030:             d = pd.read_csv(fh)
031:     except Exception as e:
032:         print(f"  skip {name}: {e}")
033:         return None
034:     if not set(NEED) <= set(d.columns):
035:         print(f"  skip {name}: not in the agreed format (columns: {list(d.columns)[:8]})")
036:         return None
037:     print(f"  ok   {name}: {len(d):,} rows, models {sorted(d['model'].astype(str).unique())}")
038:     return d[NEED]
039: 
040: 
041: def collect(folder):
042:     parts = []
043:     for path in glob.glob(os.path.join(folder, "**", "*"), recursive=True):
044:         base = os.path.basename(pa

In [ ]:

# STEP 8 - Check available prediction sources

from pathlib import Path

archive_dir = project_dir / "archive"
t29_dir = project_dir / "models" / "outputs_T29"

for label, folder in [
    ("Chronos, RF and CatBoost archives", archive_dir),
    ("LightGBM T29", t29_dir)
]:
    print(f"\n{label}:")

    if not folder.exists():
        print("DIRECTORY MISSING")
        continue

    for p in sorted(folder.rglob("*")):
        if p.is_file() and (
            p.suffix.lower() == ".zip"
            or "prediction" in p.name.lower()
        ):
            print("-", p.relative_to(project_dir))



Chronos, RF and CatBoost archives:
- archive/NE_Catboost.zip
- archive/NE_Chronos.zip
- archive/NE_RandomForest.zip
- archive/NW_Catboost.zip
- archive/NW_Chronos.zip
- archive/NW_RandomForest.zip
- archive/SE_Catboost.zip
- archive/SE_Chronos.zip
- archive/SE_RandomForest.zip
- archive/SW_Catboost.zip
- archive/SW_Chronos.zip
- archive/SW_RandomForest.zip
- archive/Task_A1_A5.zip
- archive/best_model_predictions.csv
- archive/best_model_predictions_2.csv

LightGBM T29:
- models/outputs_T29/T29_dayahead_predictions.csv
- models/outputs_T29/old outputs/F1.2_outputs_T29.zip
- models/outputs_T29/old outputs/outputs_T29_old/T29_dayahead_predictions.csv


In [ ]:

# STEP 9 - Prepare isolated prediction inputs for T36

from pathlib import Path
import shutil

input_dir = Path("/content/eval_T38_leakfree/inputs")
input_dir.mkdir(parents=True, exist_ok=True)

# Remove any previous inputs to avoid stale predictions
for p in input_dir.iterdir():
    if p.is_file():
        p.unlink()

# Original Chronos, Random Forest and CatBoost archives
zones = ["NE", "NW", "SE", "SW"]
families = ["Chronos", "RandomForest", "Catboost"]

copied = []

for zone in zones:
    for family in families:
        source = project_dir / "archive" / f"{zone}_{family}.zip"

        assert source.exists(), f"Missing archive: {source.name}"

        destination = input_dir / source.name
        shutil.copy2(source, destination)
        copied.append(destination.name)

# Current LightGBM T29 predictions only
t29_source = (
    project_dir /
    "models/outputs_T29/T29_dayahead_predictions.csv"
)

assert t29_source.exists(), "Current T29 predictions not found"

shutil.copy2(
    t29_source,
    input_dir / t29_source.name
)

# Corrected predictions from Notebooks 02 and 03
shutil.copy2(
    gbm_file,
    input_dir / "GBM_Corrected_Predictions_10series.csv"
)

shutil.copy2(
    hybrid_file,
    input_dir / "Hybrid_Corrected_Predictions_10series.csv"
)

print("INPUT DIRECTORY:", input_dir)
print("\nFiles prepared:")

for p in sorted(input_dir.iterdir()):
    print("-", p.name)

print("\nTotal input files:", len(list(input_dir.iterdir())))

assert len(list(input_dir.iterdir())) == 15

print("\nIsolated input preparation PASSED")


INPUT DIRECTORY: /content/eval_T38_leakfree/inputs

Files prepared:
- GBM_Corrected_Predictions_10series.csv
- Hybrid_Corrected_Predictions_10series.csv
- NE_Catboost.zip
- NE_Chronos.zip
- NE_RandomForest.zip
- NW_Catboost.zip
- NW_Chronos.zip
- NW_RandomForest.zip
- SE_Catboost.zip
- SE_Chronos.zip
- SE_RandomForest.zip
- SW_Catboost.zip
- SW_Chronos.zip
- SW_RandomForest.zip
- T29_dayahead_predictions.csv

Total input files: 15

Isolated input preparation PASSED


In [ ]:

# STEP 10 - Find the clean reference dataset required by T36

print("Potential clean reference datasets:\n")

clean_candidates = []

for p in sorted(project_dir.rglob("*")):
    if p.is_file() and (
        "hourly_clean" in p.name.lower()
        or "cedint_f1" in p.name.lower()
    ):
        print("-", p.relative_to(project_dir))

        if "hourly_clean" in p.name.lower():
            clean_candidates.append(p)

print("\nHourly-clean candidates found:", len(clean_candidates))


Potential clean reference datasets:

- cedint_f1_hourly_clean.csv.gz
- cleaning_and_eda/CeDInt_F1_quality_and_anomalies.xlsx
- data/T32_cedint_f1_model_ready.csv.gz
- data/cedint_f1_hourly.csv.zip
- data/cedint_f1_hourly_excel_IT_utc.zip
- data/cedint_f1_model_ready.csv
- data/cedint_f1_spatial_long.csv.gz
- data/cedint_f1_zone_environment.csv.gz
- docs_and_slides/CeDInt_F1_Results_Summary.md

Hourly-clean candidates found: 1


In [ ]:

# STEP 11 - Verify the clean reference dataset for T36

clean_file = project_dir / "cedint_f1_hourly_clean.csv.gz"

assert clean_file.exists(), "Clean reference dataset not found"

clean_df = pd.read_csv(
    clean_file,
    compression="gzip",
    nrows=1000
)

print("Clean dataset:", clean_file.name)

print("\nColumns:")
print(clean_df.columns.tolist())

print("\nFirst rows:")
display(clean_df.head(3))

required = [
    "timestamp_utc",
    "timestamp_local",
    "zone",
    "load_type",
    "energy_wh_model"
]

missing = [
    col for col in required
    if col not in clean_df.columns
]

print("\nMissing required columns:", missing)

assert not missing, (
    f"T36 reference dataset is missing: {missing}"
)

print("\nClean reference dataset verification PASSED")


Clean dataset: cedint_f1_hourly_clean.csv.gz

Columns:
['timestamp_utc', 'timestamp_local', 'zone', 'zone_code', 'load_type', 'n_devices', 'era', 'recommendation', 'usable_for_model', 'energy_wh_model', 'energy_source', 'energy_wh', 'energy_method', 'power_w', 'apparent_power_va', 'current_a', 'power_w_raw', 'counter_wh', 'counter_wh_raw', 'segment', 'temp_c', 'hum_pct', 'lum_lux', 'occupancy', 'source_rows', 'complete_rows', 'flag_missing_hour', 'flag_zero_record', 'flag_partial_sum', 'flag_counter_spike', 'flag_counter_reset', 'flag_energy_implausible', 'flag_neg_noise', 'flag_neg_large', 'reversed_sign', 'flag_low_coverage', 'flag_power_spike', 'flag_power_energy_mismatch', 'flag_env_missing', 'flag_temp_impossible', 'flag_temp_extreme', 'flag_hum_invalid', 'flag_lum_day_zero', 'series_excluded', 'is_holiday', 'holiday_name', 'is_weekend', 'is_august_low_use', 'flag_covid_2020', 'n_zero_record', 'n_partial', 'n_reset', 'n_spike', 'n_neg_noise', 'n_neg_large']

First rows:


,timestamp_utc,timestamp_local,zone,zone_code,load_type,n_devices,era,recommendation,usable_for_model,energy_wh_model,...,holiday_name,is_weekend,is_august_low_use,flag_covid_2020,n_zero_record,n_partial,n_reset,n_spike,n_neg_noise,n_neg_large
0,2020-04-02 23:00:00,2020-04-03 01:00:00,NE,1NE,HVAC,2,hourly,Use,1,3.841,...,NaN,0,0,1,0,0,0,0,0,0
1,2020-04-03 00:00:00,2020-04-03 02:00:00,NE,1NE,HVAC,2,hourly,Use,1,3.763,...,NaN,0,0,1,0,0,0,0,0,0
2,2020-04-03 01:00:00,2020-04-03 03:00:00,NE,1NE,HVAC,2,hourly,Use,1,3.683,...,NaN,0,0,1,0,0,0,0,0,0



Missing required columns: []

Clean reference dataset verification PASSED


In [ ]:

# STEP 20 - Inspect original reference baseline script

baseline_path = (
    project_dir /
    "predictions_and_eval/make_reference_baseline.py"
)

assert baseline_path.exists(), "Baseline script not found"

baseline_lines = baseline_path.read_text(
    encoding="utf-8"
).splitlines()

print("Baseline script:", baseline_path.name)
print("Total lines:", len(baseline_lines))
print("\nSCRIPT CONTENT:\n")

for i, line in enumerate(baseline_lines, start=1):
    print(f"{i:03d}: {line}")


Baseline script: make_reference_baseline.py
Total lines: 36

SCRIPT CONTENT:

001: """
002: Reference baselines in the agreed prediction format (Team B, TBDA).
003: 
004: Purpose: test the Team B evaluation toolkit (team_b_eval.py) before Team D delivers its models,
005: and give every later model a simple bar to beat. These are NOT Team D's models.
006: 
007:   naive_day      forecast for hour h = actual energy at hour h one day earlier (lag 24 h).
008:                  Valid for a day-ahead forecast issued at midnight: yesterday is fully known.
009:   naive_week     forecast for hour h = actual energy at hour h one week earlier (lag 168 h).
010: 
011: Only hours with usable_for_model = 1 are forecast; lags never cross the 902-day gap.
012: Run: python make_reference_baseline.py <clean_folder> <output_csv>
013: """
014: import sys
015: import pandas as pd
016: from team_b_eval import SPLITS, TEST_2025_ZONES, load_clean
017: 
018: clean_dir, out_csv = sys.argv[1], sys.argv[2]
019: df =

In [ ]:

# STEP 21 - Inspect T30 benchmark outputs

t30_dir = project_dir / "models" / "outputs_T30"

assert t30_dir.exists(), "T30 output directory not found"

print("T30 files:\n")
for p in sorted(t30_dir.rglob("*")):
    if p.is_file():
        print("-", p.relative_to(project_dir))

print("\nT30 forecast CSV columns:\n")

t30_csv = (
    t30_dir / "T30_floor1_total_forecast_hourly.csv"
)

if t30_csv.exists():
    sample = pd.read_csv(t30_csv, nrows=5)
    print(sample.columns.tolist())
    display(sample)


T30 files:

- models/outputs_T30/T30_coherence_gap.png
- models/outputs_T30/T30_experiment_log_rows.csv
- models/outputs_T30/T30_floor1_total_forecast_hourly.csv
- models/outputs_T30/T30_floor1_total_forecast_results.xlsx
- models/outputs_T30/T30_total_MAE_by_hour.png
- models/outputs_T30/T30_total_forecast_vs_actual.png

T30 forecast CSV columns:

['timestamp', 'actual_wh', 'seasonal_naive', 'direct_total_model', 'bottom_up', 'reconciled_ols', 'reconciled_wls', 'reconciled_mint']


,timestamp,actual_wh,seasonal_naive,direct_total_model,bottom_up,reconciled_ols,reconciled_wls,reconciled_mint
0,2022-07-01T00:00:00Z,1639.856,1600.9715,1908.196569,1731.192068,1930.387572,1888.055584,1861.439995
1,2022-07-01T01:00:00Z,1625.924,1519.4445,1846.781062,1699.241031,1831.030028,1792.725848,1808.293875
2,2022-07-01T02:00:00Z,1583.381,1479.7885,1806.527263,1729.150985,1802.800638,1784.649186,1825.575423
3,2022-07-01T03:00:00Z,1502.840,1445.6550,1805.536992,1727.326356,1807.988770,1791.859509,1830.640994
4,2022-07-01T04:00:00Z,1525.153,1462.5860,1875.278328,1744.799917,1851.914795,1816.503800,1811.415286


In [ ]:

# STEP 22 - Generate original T38 reference baselines

import subprocess
import sys
import pandas as pd

baseline_script = (
    project_dir /
    "predictions_and_eval/make_reference_baseline.py"
)

baseline_output = (
    input_dir / "Reference_Baseline_predictions.csv"
)

assert baseline_script.exists()

command = [
    sys.executable,
    str(baseline_script),
    str(project_dir),
    str(baseline_output)
]

result = subprocess.run(
    command,
    capture_output=True,
    text=True
)

print("RETURN CODE:", result.returncode)
print("\nOUTPUT:")
print(result.stdout)

if result.stderr:
    print("\nERRORS:")
    print(result.stderr[-4000:])

assert result.returncode == 0, "Baseline generation failed"
assert baseline_output.exists(), "Baseline CSV not created"

baseline_df = pd.read_csv(baseline_output)

print("\nModels generated:")
print(baseline_df["model"].value_counts())

print("\nEnergy series:")
print(
    baseline_df.groupby("model").apply(
        lambda g: g.groupby(
            ["zone", "load_type"]
        ).ngroups,
        include_groups=False
    )
)

print("\nBaseline generation PASSED")


RETURN CODE: 0

OUTPUT:
726713 rows written to /content/eval_T38_leakfree/inputs/Reference_Baseline_predictions.csv


Models generated:
model
naive_day     367661
naive_week    359052
Name: count, dtype: int64

Energy series:
model
naive_day     17
naive_week    17
dtype: int64

Baseline generation PASSED


In [ ]:

# STEP 12 - Verify all 16 prediction input files

input_files = sorted(
    p.name for p in input_dir.iterdir()
    if p.is_file()
)

print("Total input files:", len(input_files))

assert len(input_files) == 16, (
    f"Expected 16 files, found {len(input_files)}"
)

baseline_file = (
    input_dir / "Reference_Baseline_predictions.csv"
)

assert baseline_file.exists(), "Baseline file missing"

baseline_check = pd.read_csv(baseline_file)

assert {"naive_day", "naive_week"}.issubset(
    set(baseline_check["model"].unique())
)

print("Reference baselines verified")
print("Input verification PASSED")


Total input files: 16
Reference baselines verified
Input verification PASSED


In [ ]:

# STEP 13 - Run official T36 on isolated prediction inputs

import subprocess
import sys
from pathlib import Path

t36_output = Path(
    "/content/eval_T38_leakfree/all_model_predictions.csv"
)

# T36 expects the clean dataset directory, not the file path

command = [
    sys.executable,
    str(t36_path),
    str(input_dir),
    str(t36_output),
    str(project_dir)
]


print("Running official T36...\n")

result = subprocess.run(
    command,
    capture_output=True,
    text=True
)

print("RETURN CODE:", result.returncode)

print("\nT36 OUTPUT:")
print(result.stdout[-12000:])

if result.stderr:
    print("\nT36 ERRORS OR WARNINGS:")
    print(result.stderr[-4000:])

if result.returncode == 0:
    print("\nT36 execution completed")
    print("Output file exists:", t36_output.exists())
else:
    print("\nT36 execution failed - inspect errors above")


Running official T36...

RETURN CODE: 0

T36 OUTPUT:
Searching /content/eval_T38_leakfree/inputs ...
  ok   NW_Chronos.zip/NW_RACK_chronos_bolt_base_predictions.csv: 6,670 rows, models ['chronos_bolt_base']
  ok   NW_Chronos.zip/NW_Plugs_Sockets_chronos_bolt_base_predictions.csv: 6,713 rows, models ['chronos_bolt_base']
  ok   NW_Chronos.zip/NW_Emerg_Lighting_chronos_bolt_base_predictions.csv: 6,075 rows, models ['chronos_bolt_base']
  ok   NW_Chronos.zip/NW_HVAC_chronos_bolt_base_predictions.csv: 6,053 rows, models ['chronos_bolt_base']
  ok   NW_Chronos.zip/NW_Lighting_chronos_bolt_base_predictions.csv: 6,712 rows, models ['chronos_bolt_base']
  ok   SW_Catboost.zip/SW_Plugs_Sockets_CatBoost_predictions.csv: 8,946 rows, models ['catboost', 'catboost_no2020']
  ok   SW_Catboost.zip/SW_Emerg_Lighting_CatBoost_predictions.csv: 8,636 rows, models ['catboost', 'catboost_no2020']
  ok   SW_Catboost.zip/SW_Lighting_CatBoost_predictions.csv: 8,718 rows, models ['catboost', 'catboost_no2020']

In [ ]:

# STEP 14 - Audit T36 combined output

assert t36_output.exists(), "T36 output file not found"

combined = pd.read_csv(t36_output)

print("Total prediction rows:", len(combined))

print("\nModels included:")
display(
    combined.groupby("model")
    .agg(
        rows=("model", "size"),
        series=("zone", lambda x: 0)
    )
    .drop(columns="series")
    .reset_index()
)

print("\nPrediction coverage by model:")
display(
    combined.groupby("model")
    .apply(
        lambda g: g.groupby(
            ["zone", "load_type"]
        ).ngroups,
        include_groups=False
    )
    .reset_index(name="energy_series")
)

print("\nPrediction splits:")
display(
    combined.groupby(["model", "split"])
    .size()
    .unstack(fill_value=0)
)

print("\nGBM and Hybrid variants:")
print(sorted([
    m for m in combined["model"].unique()
    if "gbm" in m.lower() or "hybrid" in m.lower()
]))


Total prediction rows: 806964

Models included:


,model,rows
0,catboost,83221
1,catboost_no2020,83221
2,chronos_bolt_base,82731
3,gbm_leakfree,46007
4,hybrid_leakfree,46007
5,lightgbm,60087
6,naive_day,93832
7,naive_week,89757
8,random_forest,83221
9,random_forest_no2020,83221



Prediction coverage by model:


,model,energy_series
0,catboost,14
1,catboost_no2020,14
2,chronos_bolt_base,14
3,gbm_leakfree,10
4,hybrid_leakfree,10
5,lightgbm,10
6,naive_day,16
7,naive_week,16
8,random_forest,14
9,random_forest_no2020,14



Prediction splits:


split,test_2023,test_2025,validation
model,,,
catboost,17185,13470,52566
catboost_no2020,17185,13470,52566
chronos_bolt_base,17119,13169,52443
gbm_leakfree,9394,7089,29524
hybrid_leakfree,9394,7089,29524
lightgbm,12516,9825,37746
naive_day,19696,13334,60802
naive_week,19405,11910,58442
random_forest,17185,13470,52566



GBM and Hybrid variants:
['gbm_leakfree', 'hybrid_leakfree', 'lightgbm']


In [ ]:

# STEP 24 - Verify official benchmarks in T36 output

combined = pd.read_csv(t36_output)

models = set(combined["model"].unique())

required_models = {
    "naive_day",
    "naive_week",
    "chronos_bolt_base",
    "gbm_leakfree",
    "hybrid_leakfree",
    "lightgbm"
}

missing = required_models - models

print("Models:", sorted(models))
print("Missing required models:", sorted(missing))

assert not missing

print("\nBenchmark coverage:")
display(
    combined[
        combined["model"].isin(["naive_day", "naive_week"])
    ].groupby(["model", "split"])
    .size()
    .unstack(fill_value=0)
)

print("\nBenchmark verification PASSED")


Models: ['catboost', 'catboost_no2020', 'chronos_bolt_base', 'gbm_leakfree', 'hybrid_leakfree', 'lightgbm', 'naive_day', 'naive_week', 'random_forest', 'random_forest_no2020', 'seasonalnaive']
Missing required models: []

Benchmark coverage:


split,test_2023,test_2025,validation
model,,,
naive_day,19696,13334,60802
naive_week,19405,11910,58442



Benchmark verification PASSED


In [ ]:

# STEP 18 - Locate original benchmark predictions

from pathlib import Path

keywords = [
    "naive",
    "baseline",
    "benchmark",
    "t30"
]

print("POTENTIAL BENCHMARK FILES\n")

for p in sorted(project_dir.rglob("*")):
    if p.is_file():
        name = p.name.lower()

        if any(word in name for word in keywords):
            print("-", p.relative_to(project_dir))

print("\nT38 BENCHMARK DEFINITIONS\n")

for i, line in enumerate(t38_lines):
    if any(word in line.lower() for word in [
        "naive_day",
        "naive_week",
        "seasonalnaive",
        "skill"
    ]):
        print(f"{i+1:03d}: {line}")


POTENTIAL BENCHMARK FILES

- figures/F11_error_by_hour_naive_day.png
- figures/F13_best_model_vs_benchmark.png
- models/F1.3_T30_floor1_total_forecast_MH.R
- models/outputs_T30/T30_coherence_gap.png
- models/outputs_T30/T30_experiment_log_rows.csv
- models/outputs_T30/T30_floor1_total_forecast_hourly.csv
- models/outputs_T30/T30_floor1_total_forecast_results.xlsx
- models/outputs_T30/T30_total_MAE_by_hour.png
- models/outputs_T30/T30_total_forecast_vs_actual.png
- predictions_and_eval/make_reference_baseline.py

T38 BENCHMARK DEFINITIONS

005:   * skill score per series and period: 1 - RMSE_model / RMSE_naive (naive = same hour yesterday)
031: BENCH = {"naive_day", "naive_week", "seasonalnaive"}
048: def skill(rm, rn):
071:         return "above 50 %" if cv > 50 else ("bias > 10 %" if abs(nmbe) > 10 else "no skill")
114:         pool = v[v["model"].isin(cands + ["naive_day"])]
121:                    sel_cv_best=sc[best]["CV"], sel_cv_naive_day=sc.get("naive_day", {}).get("CV", np.nan)

In [ ]:

# STEP 19 - Inspect benchmark and evaluation logic in T38

print("T38 SETUP")
for i in range(20, min(65, len(t38_lines))):
    print(f"{i+1:03d}: {t38_lines[i]}")

print("\nT38 SELECTION AND METRICS")
for i in range(108, min(162, len(t38_lines))):
    print(f"{i+1:03d}: {t38_lines[i]}")


T38 SETUP
021: Run:
022:   python TeamB_T38_final_evaluation.py all_model_predictions.csv <clean_folder> <T30_hourly_csv> <out_folder>
023: Inputs:  all_model_predictions.csv from T36_combine_predictions.py (with the clean-data timestamp check),
024:          cedint_f1_hourly_clean.csv.gz (for local dates), outputs_T30/T30_floor1_total_forecast_hourly.csv
025: Outputs: TeamB_T38_final_evaluation.xlsx, best_model_selection_final.csv, best_model_predictions_final.csv
026: """
027: import os, sys
028: import numpy as np
029: import pandas as pd
030: 
031: BENCH = {"naive_day", "naive_week", "seasonalnaive"}
032: SEL = ("2022-10-16 00:00", "2022-12-31 23:00")          # out-of-sample half of validation for every model
033: SPLITS = ["validation", "test_2023", "test_2025"]
034: 
035: 
036: def m(y, f):
037:     y, f = np.asarray(y, float), np.asarray(f, float)
038:     ok = ~np.isnan(y) & ~np.isnan(f)
039:     y, f = y[ok], f[ok]
040:     if len(y) == 0 or y.mean() == 0:
041:         return

In [ ]:

# STEP 27 - Install Excel export dependency

%pip install -q XlsxWriter

import xlsxwriter

print("XlsxWriter installed successfully!")
print("Version:", xlsxwriter.__version__)


XlsxWriter installed successfully!
Version: 3.3.4


In [ ]:

# STEP 25 - Run official T38 final model evaluation

import subprocess
import sys
from pathlib import Path

t38_script = (
    project_dir /
    "predictions_and_eval/TeamB_T38_final_evaluation.py"
)

t30_file = (
    project_dir /
    "models/outputs_T30/T30_floor1_total_forecast_hourly.csv"
)

t38_output_dir = Path(
    "/content/eval_T38_leakfree/t38_results"
)

t38_output_dir.mkdir(parents=True, exist_ok=True)

assert t38_script.exists(), "T38 script missing"
assert t36_output.exists(), "Combined predictions missing"
assert t30_file.exists(), "T30 hourly predictions missing"
assert clean_file.exists(), "Clean dataset missing"

# Official T38 expects the clean DATASET DIRECTORY
command = [
    sys.executable,
    str(t38_script),
    str(t36_output),
    str(project_dir),
    str(t30_file),
    str(t38_output_dir)
]

print("Running official T38 evaluation...")

result_t38 = subprocess.run(
    command,
    capture_output=True,
    text=True
)

print("\nRETURN CODE:", result_t38.returncode)

print("\nT38 OUTPUT:")
print(result_t38.stdout[-10000:])

if result_t38.stderr:
    print("\nERRORS OR WARNINGS:")
    print(result_t38.stderr[-5000:])

if result_t38.returncode == 0:
    print("\nT38 execution completed")
    print("\nGenerated files:")

    for p in sorted(t38_output_dir.rglob("*")):
        if p.is_file():
            print("-", p.name)
else:
    print("\nT38 execution failed: inspect the error above")


Running official T38 evaluation...

RETURN CODE: 0

T38 OUTPUT:
               model  series_won
   chronos_bolt_base          10
     hybrid_leakfree           2
random_forest_no2020           1
            catboost           1
     split  series  positive_skill  median_skill
 test_2023      14              14          0.30
 test_2025      11              11          0.20
validation      14              14          0.27
     split      load_group  series  positive_skill  median_skill  on_target  median_daily_CV
 test_2023        Constant       4               4          0.37          4             2.11
 test_2023            HVAC       3               3          0.29          2            28.40
 test_2023        Lighting       3               3          0.37          0            99.13
 test_2023 Plugs & Sockets       4               4          0.24          2            24.20
 test_2025        Constant       3               3          0.29          2             4.85
 test_2025       

In [ ]:

# STEP 26 - Verify T38 output files

assert result_t38.returncode == 0, (
    "T38 did not complete successfully"
)

output_files = [
    p for p in t38_output_dir.rglob("*")
    if p.is_file()
]

print("Generated files:", len(output_files))

for p in sorted(output_files):
    print("-", p.relative_to(t38_output_dir))

assert output_files, "T38 produced no output files"

print("\nT38 output verification PASSED")


Generated files: 3
- TeamB_T38_final_evaluation.xlsx
- best_model_predictions_final.csv
- best_model_selection_final.csv

T38 output verification PASSED


In [ ]:

# STEP 29 - Inspect updated T38 model selection

selection_file = (
    t38_output_dir / "best_model_selection_final.csv"
)

assert selection_file.exists(), "Selection CSV not found"

selection_new = pd.read_csv(selection_file)

print("Selection columns:")
print(selection_new.columns.tolist())

print("\nNumber of selected series:", len(selection_new))

assert len(selection_new) == 14, (
    "Expected 14 energy series"
)

print("\nUPDATED MODEL SELECTION:")
display(
    selection_new[
        ["zone", "load_type", "best_model"]
    ].sort_values(["zone", "load_type"])
)

print("\nWINNING MODELS:")
print(selection_new["best_model"].value_counts())


Selection columns:
['zone', 'load_type', 'best_model', 'candidates', 'selection_hours', 'sel_cv_best', 'sel_cv_naive_day', 'sel_cv_catboost', 'sel_cv_catboost_no2020', 'sel_cv_chronos_bolt_base', 'sel_cv_gbm_leakfree', 'sel_cv_hybrid_leakfree', 'sel_cv_lightgbm', 'sel_cv_random_forest', 'sel_cv_random_forest_no2020']

Number of selected series: 14

UPDATED MODEL SELECTION:


,zone,load_type,best_model
0,NE,HVAC,random_forest_no2020
1,NE,Plugs & Sockets,hybrid_leakfree
2,NW,Emerg. Lighting,chronos_bolt_base
3,NW,HVAC,chronos_bolt_base
4,NW,Lighting,hybrid_leakfree
5,NW,Plugs & Sockets,catboost
6,NW,RACK,chronos_bolt_base
7,SE,Emerg. Lighting,chronos_bolt_base
8,SE,HVAC,chronos_bolt_base
9,SE,Lighting,chronos_bolt_base



WINNING MODELS:
best_model
chronos_bolt_base       10
hybrid_leakfree          2
random_forest_no2020     1
catboost                 1
Name: count, dtype: int64


In [ ]:

# STEP 30 - Compare original vs updated T38 selection

original_file = (
    project_dir /
    "predictions_and_eval/best_model_selection_final.csv"
)

assert original_file.exists(), "Original T38 selection missing"

selection_original = pd.read_csv(original_file)

original = selection_original[
    ["zone", "load_type", "best_model"]
].rename(columns={
    "best_model": "original_winner"
})

updated = selection_new[
    ["zone", "load_type", "best_model"]
].rename(columns={
    "best_model": "updated_winner"
})

selection_changes = original.merge(
    updated,
    on=["zone", "load_type"],
    how="outer",
    validate="one_to_one"
)

selection_changes["changed"] = (
    selection_changes["original_winner"] !=
    selection_changes["updated_winner"]
)

print("MODEL SELECTION COMPARISON:")
display(
    selection_changes.sort_values(
        ["zone", "load_type"]
    )
)

print("\nChanged selections:",
      selection_changes["changed"].sum())

print("\nOriginal winner counts:")
print(selection_changes["original_winner"].value_counts())

print("\nUpdated winner counts:")
print(selection_changes["updated_winner"].value_counts())


MODEL SELECTION COMPARISON:


,zone,load_type,original_winner,updated_winner,changed
0,NE,HVAC,gbm,random_forest_no2020,True
1,NE,Plugs & Sockets,gbm,hybrid_leakfree,True
2,NW,Emerg. Lighting,chronos_bolt_base,chronos_bolt_base,False
3,NW,HVAC,chronos_bolt_base,chronos_bolt_base,False
4,NW,Lighting,hybrid_chronos_gbm_learned_weights,hybrid_leakfree,True
5,NW,Plugs & Sockets,gbm,catboost,True
6,NW,RACK,chronos_bolt_base,chronos_bolt_base,False
7,SE,Emerg. Lighting,chronos_bolt_base,chronos_bolt_base,False
8,SE,HVAC,chronos_bolt_base,chronos_bolt_base,False
9,SE,Lighting,hybrid_chronos_gbm_learned_weights,chronos_bolt_base,True



Changed selections: 7

Original winner counts:
original_winner
chronos_bolt_base                     7
gbm                                   3
hybrid_chronos_gbm_learned_weights    3
random_forest_no2020                  1
Name: count, dtype: int64

Updated winner counts:
updated_winner
chronos_bolt_base       10
hybrid_leakfree          2
random_forest_no2020     1
catboost                 1
Name: count, dtype: int64


In [ ]:

# STEP 31 - Audit selection hours and CV(RMSE)

keys = ["zone", "load_type"]

audit_original = selection_original[
    keys + ["best_model", "selection_hours", "sel_cv_best"]
].rename(columns={
    "best_model": "original_winner",
    "selection_hours": "original_hours",
    "sel_cv_best": "original_cv"
})

audit_updated = selection_new[
    keys + ["best_model", "selection_hours", "sel_cv_best",
            "sel_cv_naive_day"]
].rename(columns={
    "best_model": "updated_winner",
    "selection_hours": "updated_hours",
    "sel_cv_best": "updated_cv",
    "sel_cv_naive_day": "naive_day_cv"
})

selection_audit = audit_original.merge(
    audit_updated,
    on=keys,
    how="outer",
    validate="one_to_one"
)

selection_audit["hours_difference"] = (
    selection_audit["updated_hours"] -
    selection_audit["original_hours"]
)

print("T38 SELECTION AUDIT")
display(
    selection_audit.sort_values(keys).round(2)
)

print("\nMissing benchmark scores:",
      selection_audit["naive_day_cv"].isna().sum())

print("\nMissing winner scores:",
      selection_audit["updated_cv"].isna().sum())

print("\nSeries with changed selection hours:",
      (selection_audit["hours_difference"] != 0).sum())

assert len(selection_audit) == 14

print("\nSelection coverage audit completed")


T38 SELECTION AUDIT


,zone,load_type,original_winner,original_hours,original_cv,updated_winner,updated_hours,updated_cv,naive_day_cv,hours_difference
0,NE,HVAC,gbm,1813,325.35,random_forest_no2020,1810,326.59,462.30,-3
1,NE,Plugs & Sockets,gbm,1813,23.24,hybrid_leakfree,1810,41.91,50.45,-3
2,NW,Emerg. Lighting,chronos_bolt_base,1630,2.70,chronos_bolt_base,1630,2.70,3.46,0
3,NW,HVAC,chronos_bolt_base,1606,19.12,chronos_bolt_base,1603,19.10,23.52,-3
4,NW,Lighting,hybrid_chronos_gbm_learned_weights,1813,64.35,hybrid_leakfree,1810,64.39,95.51,-3
5,NW,Plugs & Sockets,gbm,1813,37.88,catboost,1810,48.28,65.55,-3
6,NW,RACK,chronos_bolt_base,1837,0.74,chronos_bolt_base,1837,0.74,1.03,0
7,SE,Emerg. Lighting,chronos_bolt_base,1612,12.74,chronos_bolt_base,1612,12.74,19.93,0
8,SE,HVAC,chronos_bolt_base,1136,21.86,chronos_bolt_base,1108,23.24,44.19,-28
9,SE,Lighting,hybrid_chronos_gbm_learned_weights,1071,193.67,chronos_bolt_base,1038,211.59,277.35,-33



Missing benchmark scores: 0

Missing winner scores: 0

Series with changed selection hours: 10

Selection coverage audit completed


In [ ]:

# STEP 32 - Compare original and updated selection skill scores

import numpy as np
import pandas as pd

required = [
    "zone", "load_type", "sel_cv_best",
    "sel_cv_naive_day", "selection_hours"
]

for name, table in [
    ("Original", selection_original),
    ("Updated", selection_new)
]:
    missing = [c for c in required if c not in table.columns]
    assert not missing, f"{name} missing columns: {missing}"

def calculate_skill(table):
    result = table.copy()

    best = pd.to_numeric(result["sel_cv_best"], errors="coerce")
    naive = pd.to_numeric(result["sel_cv_naive_day"], errors="coerce")

    result["skill_score"] = np.where(
        naive > 0,
        1 - best / naive,
        np.nan
    )

    return result

original_skill = calculate_skill(selection_original)
updated_skill = calculate_skill(selection_new)

print("ORIGINAL T38 - SEL")
print("Median SEL skill:", round(original_skill["skill_score"].median(), 4))
print("Positive skill:", (original_skill["skill_score"] > 0).sum(), "/ 14")

print("\nUPDATED T38 - SEL")
print("Median SEL skill:", round(updated_skill["skill_score"].median(), 4))
print("Positive skill:", (updated_skill["skill_score"] > 0).sum(), "/ 14")

skill_comparison = original_skill[
    ["zone", "load_type", "skill_score"]
].rename(columns={"skill_score": "original_skill"}).merge(
    updated_skill[
        ["zone", "load_type", "skill_score"]
    ].rename(columns={"skill_score": "updated_skill"}),
    on=["zone", "load_type"],
    validate="one_to_one"
)

skill_comparison["difference"] = (
    skill_comparison["updated_skill"] -
    skill_comparison["original_skill"]
)

print("\nSEL SKILL SCORE COMPARISON")
display(
    skill_comparison.sort_values(["zone", "load_type"]).round(4)
)

print("\nMissing skill values:",
      skill_comparison[["original_skill", "updated_skill"]]
      .isna().sum().sum())

assert len(skill_comparison) == 14

print("\nSkill Score audit completed")


ORIGINAL T38 - SEL
Median SEL skill: 0.3423
Positive skill: 14 / 14

UPDATED T38 - SEL
Median SEL skill: 0.2503
Positive skill: 14 / 14

SEL SKILL SCORE COMPARISON


,zone,load_type,original_skill,updated_skill,difference
0,NE,HVAC,0.2944,0.2936,-0.0008
1,NE,Plugs & Sockets,0.5382,0.1693,-0.3690
2,NW,Emerg. Lighting,0.2197,0.2197,0.0000
3,NW,HVAC,0.1881,0.1879,-0.0002
4,NW,Lighting,0.3091,0.3258,0.0167
5,NW,Plugs & Sockets,0.4080,0.2635,-0.1446
6,NW,RACK,0.2816,0.2816,0.0000
7,SE,Emerg. Lighting,0.3608,0.3608,0.0000
8,SE,HVAC,0.4703,0.4741,0.0038
9,SE,Lighting,0.4813,0.2371,-0.2441



Missing skill values: 0

Skill Score audit completed


In [ ]:

# STEP 33 - Inspect official T38 evaluation outputs

from pathlib import Path
import pandas as pd

evaluation_file = (
    t38_output_dir / "TeamB_T38_final_evaluation.xlsx"
)

predictions_file = (
    t38_output_dir / "best_model_predictions_final.csv"
)

assert evaluation_file.exists()
assert predictions_file.exists()

# Inspect the Excel workbook structure
excel_file = pd.ExcelFile(evaluation_file)

print("T38 EXCEL SHEETS:")
for sheet in excel_file.sheet_names:
    print("-", sheet)

# Inspect final prediction data
final_predictions = pd.read_csv(predictions_file)

print("\nFINAL PREDICTION COLUMNS:")
print(final_predictions.columns.tolist())

print("\nPREDICTIONS BY PERIOD:")
if "split" in final_predictions.columns:
    print(final_predictions["split"].value_counts())

print("\nEXCEL SHEET PREVIEWS:")
for sheet in excel_file.sheet_names:
    data = pd.read_excel(
        evaluation_file,
        sheet_name=sheet
    )
    print(f"\n{sheet}: {len(data)} rows")
    print("Columns:", data.columns.tolist())
    display(data.head(3))


T38 EXCEL SHEETS:
- README
- Summary
- Summary_total
- Final_table
- Model_choice
- Model_wins
- All_models
- Floor_total

FINAL PREDICTION COLUMNS:
['timestamp_utc', 'zone', 'load_type', 'model', 'split', 'y_true', 'y_pred', 'source_model']

PREDICTIONS BY PERIOD:
split
validation    52078
test_2023     16952
test_2025     12776
Name: count, dtype: int64

EXCEL SHEET PREVIEWS:

README: 11 rows
Columns: ['item', 'description']


,item,description
0,Purpose,T38 (F6.4): final evaluation of every day-ahea...
1,Model choice,"Per series, the model with the lowest CV(RMSE)..."
2,Skill score,1 - RMSE(model) / RMSE(naive same hour yesterd...



Summary: 12 rows
Columns: ['split', 'load_group', 'series', 'positive_skill', 'median_skill', 'on_target', 'median_daily_CV']


,split,load_group,series,positive_skill,median_skill,on_target,median_daily_CV
0,test_2023,Constant,4,4,0.368,4,2.111
1,test_2023,HVAC,3,3,0.293,2,28.397
2,test_2023,Lighting,3,3,0.374,0,99.128



Summary_total: 3 rows
Columns: ['split', 'series', 'positive_skill', 'median_skill']


,split,series,positive_skill,median_skill
0,test_2023,14,14,0.296
1,test_2025,11,11,0.201
2,validation,14,14,0.266


... (6 outputs left out)


All_models: 393 rows
Columns: ['zone', 'load_type', 'split', 'model', 'hours', 'CV', 'NMBE', 'MAE_kWh_per_h', 'skill']


,zone,load_type,split,model,hours,CV,NMBE,MAE_kWh_per_h,skill
0,NE,HVAC,test_2023,catboost,1389,691.726,61.581,0.005,0.293
1,NE,HVAC,test_2023,catboost_no2020,1389,691.617,60.620,0.005,0.293
2,NE,HVAC,test_2023,chronos_bolt_base,1389,691.065,57.037,0.004,0.293



Floor_total: 8 rows
Columns: ['split', 'method', 'hours', 'CV', 'NMBE', 'MAE_kWh_per_h', 'skill_vs_seasonal_naive', 'days', 'daily_CV', 'daily_NMBE', 'daily_MAE_kWh']


,split,method,hours,CV,NMBE,MAE_kWh_per_h,skill_vs_seasonal_naive,days,daily_CV,daily_NMBE,daily_MAE_kWh
0,test_2023,seasonal_naive,586,45.205,-6.300,0.994,0.000,11,33.55,-7.644,12.784
1,test_2023,direct_total_model,586,38.641,-7.909,0.969,0.145,11,35.90,-22.921,19.703
2,test_2023,bottom_up,586,37.982,-7.418,0.966,0.160,11,31.85,-19.214,17.103


In [ ]:

# STEP 34 - Summarize updated T38 test performance

summary_total = pd.read_excel(
    evaluation_file,
    sheet_name="Summary_total"
)

summary_detail = pd.read_excel(
    evaluation_file,
    sheet_name="Summary"
)

final_table = pd.read_excel(
    evaluation_file,
    sheet_name="Final_table"
)

print("UPDATED SUMMARY BY PERIOD")
display(summary_total)

print("\nUPDATED SUMMARY BY LOAD GROUP")
display(summary_detail)

print("\nFINAL TABLE - TEST PERIODS")
display(
    final_table[
        final_table["split"].isin(
            ["test_2023", "test_2025"]
        )
    ][
        ["zone", "load_type", "split",
         "best_model", "CV", "NMBE", "skill"]
    ].sort_values(
        ["split", "zone", "load_type"]
    ).round(3)
)


UPDATED SUMMARY BY PERIOD


,split,series,positive_skill,median_skill
0,test_2023,14,14,0.296
1,test_2025,11,11,0.201
2,validation,14,14,0.266



UPDATED SUMMARY BY LOAD GROUP


,split,load_group,series,positive_skill,median_skill,on_target,median_daily_CV
0,test_2023,Constant,4,4,0.368,4,2.111
1,test_2023,HVAC,3,3,0.293,2,28.397
2,test_2023,Lighting,3,3,0.374,0,99.128
3,test_2023,Plugs & Sockets,4,4,0.239,2,24.202
4,test_2025,Constant,3,3,0.291,2,4.852
5,test_2025,HVAC,3,3,0.155,2,17.410
6,test_2025,Lighting,2,2,0.171,0,94.627
7,test_2025,Plugs & Sockets,3,3,0.198,2,15.970
8,validation,Constant,4,4,0.287,4,1.518
9,validation,HVAC,3,3,0.259,1,48.927



FINAL TABLE - TEST PERIODS


,zone,load_type,split,best_model,CV,NMBE,skill
1,NE,HVAC,test_2023,random_forest_no2020,690.320,59.736,0.293
4,NE,Plugs & Sockets,test_2023,hybrid_leakfree,20.982,0.914,0.213
7,NW,Emerg. Lighting,test_2023,chronos_bolt_base,5.314,0.823,0.568
10,NW,HVAC,test_2023,chronos_bolt_base,39.294,4.208,0.239
13,NW,Lighting,test_2023,hybrid_leakfree,61.653,7.102,0.405
16,NW,Plugs & Sockets,test_2023,catboost,53.204,-0.375,0.193
19,NW,RACK,test_2023,chronos_bolt_base,0.510,-0.003,0.387
22,SE,Emerg. Lighting,test_2023,chronos_bolt_base,12.444,-1.244,0.350
25,SE,HVAC,test_2023,chronos_bolt_base,39.992,3.746,0.300
28,SE,Lighting,test_2023,chronos_bolt_base,158.378,23.446,0.249


In [ ]:

# STEP 35 - Audit updated T38 project targets

import pandas as pd

summary_detail = pd.read_excel(
    evaluation_file,
    sheet_name="Summary"
)

print("UPDATED PROJECT TARGETS")
print("\nAvailable columns:")
print(summary_detail.columns.tolist())

required = [
    "split",
    "load_group",
    "series",
    "positive_skill",
    "median_skill",
    "on_target"
]

assert all(
    col in summary_detail.columns
    for col in required
), "Some target columns are missing"

target_summary = (
    summary_detail.groupby("split", as_index=False)
    .agg(
        series=("series", "sum"),
        positive_skill=("positive_skill", "sum"),
        on_target=("on_target", "sum")
    )
)

print("\nTARGET SUMMARY BY PERIOD")
display(target_summary)

print("\nTARGETS BY LOAD GROUP")
display(
    summary_detail[
        required
    ].sort_values(["split", "load_group"])
)

print("\nTarget audit completed")


UPDATED PROJECT TARGETS

Available columns:
['split', 'load_group', 'series', 'positive_skill', 'median_skill', 'on_target', 'median_daily_CV']

TARGET SUMMARY BY PERIOD


,split,series,positive_skill,on_target
0,test_2023,14,14,8
1,test_2025,11,11,6
2,validation,14,14,7



TARGETS BY LOAD GROUP


,split,load_group,series,positive_skill,median_skill,on_target
0,test_2023,Constant,4,4,0.368,4
1,test_2023,HVAC,3,3,0.293,2
2,test_2023,Lighting,3,3,0.374,0
3,test_2023,Plugs & Sockets,4,4,0.239,2
4,test_2025,Constant,3,3,0.291,2
5,test_2025,HVAC,3,3,0.155,2
6,test_2025,Lighting,2,2,0.171,0
7,test_2025,Plugs & Sockets,3,3,0.198,2
8,validation,Constant,4,4,0.287,4
9,validation,HVAC,3,3,0.259,1



Target audit completed


In [ ]:

# STEP 36 - Locate original T38 evaluation workbook

from pathlib import Path

print("ORIGINAL T38 EVALUATION FILES\n")

original_workbooks = []

for path in sorted(project_dir.rglob("*.xlsx")):
    if (
        "t38" in path.name.lower()
        or "final_evaluation" in path.name.lower()
    ):
        original_workbooks.append(path)
        print("-", path.relative_to(project_dir))

print("\nWorkbooks found:", len(original_workbooks))


ORIGINAL T38 EVALUATION FILES

- predictions_and_eval/TeamB_T38_final_evaluation.xlsx

Workbooks found: 1


In [ ]:

# STEP 37 - Original vs updated T38 evaluation

original_excel = (
    project_dir /
    "predictions_and_eval/TeamB_T38_final_evaluation.xlsx"
)

assert original_excel.exists()

original_summary = pd.read_excel(
    original_excel,
    sheet_name="Summary"
)

original_total = pd.read_excel(
    original_excel,
    sheet_name="Summary_total"
)

updated_summary = pd.read_excel(
    evaluation_file,
    sheet_name="Summary"
)

updated_total = pd.read_excel(
    evaluation_file,
    sheet_name="Summary_total"
)

# Compare project targets
def summarize_targets(data):
    return (
        data.groupby("split", as_index=False)
        .agg(
            series=("series", "sum"),
            positive_skill=("positive_skill", "sum"),
            on_target=("on_target", "sum")
        )
    )

before = summarize_targets(original_summary).rename(
    columns={
        "series": "original_series",
        "positive_skill": "original_positive",
        "on_target": "original_targets"
    }
)

after = summarize_targets(updated_summary).rename(
    columns={
        "series": "updated_series",
        "positive_skill": "updated_positive",
        "on_target": "updated_targets"
    }
)

target_comparison = before.merge(
    after, on="split", validate="one_to_one"
)

target_comparison["target_difference"] = (
    target_comparison["updated_targets"] -
    target_comparison["original_targets"]
)

print("ORIGINAL VS UPDATED PROJECT TARGETS")
display(target_comparison)

# Compare aggregate metrics
aggregate_comparison = original_total.merge(
    updated_total,
    on="split",
    suffixes=("_original", "_updated"),
    validate="one_to_one"
)

print("\nORIGINAL VS UPDATED AGGREGATE METRICS")
display(aggregate_comparison.round(4))

print("\nT38 comparison completed")


ORIGINAL VS UPDATED PROJECT TARGETS


,split,original_series,original_positive,original_targets,updated_series,updated_positive,updated_targets,target_difference
0,test_2023,14,14,8,14,14,8,0
1,test_2025,11,9,5,11,11,6,1
2,validation,14,14,7,14,14,7,0



ORIGINAL VS UPDATED AGGREGATE METRICS


,split,series_original,positive_skill_original,median_skill_original,series_updated,positive_skill_updated,median_skill_updated
0,test_2023,14,14,0.296,14,14,0.296
1,test_2025,11,9,0.189,11,11,0.201
2,validation,14,14,0.307,14,14,0.266



T38 comparison completed


In [ ]:

# STEP 38 - Compare targets by load group

keys = ["split", "load_group"]

group_before = original_summary[
    keys + ["series", "on_target"]
].rename(columns={
    "series": "original_series",
    "on_target": "original_targets"
})

group_after = updated_summary[
    keys + ["series", "on_target"]
].rename(columns={
    "series": "updated_series",
    "on_target": "updated_targets"
})

group_comparison = group_before.merge(
    group_after,
    on=keys,
    validate="one_to_one"
)

group_comparison["difference"] = (
    group_comparison["updated_targets"] -
    group_comparison["original_targets"]
)

print("TARGET COMPARISON BY LOAD GROUP")
display(
    group_comparison.sort_values(
        ["split", "load_group"]
    )
)


TARGET COMPARISON BY LOAD GROUP


,split,load_group,original_series,original_targets,updated_series,updated_targets,difference
0,test_2023,Constant,4,4,4,4,0
1,test_2023,HVAC,3,2,3,2,0
2,test_2023,Lighting,3,0,3,0,0
3,test_2023,Plugs & Sockets,4,2,4,2,0
4,test_2025,Constant,3,2,3,2,0
5,test_2025,HVAC,3,2,3,2,0
6,test_2025,Lighting,2,0,2,0,0
7,test_2025,Plugs & Sockets,3,1,3,2,1
8,validation,Constant,4,4,4,4,0
9,validation,HVAC,3,1,3,1,0


In [ ]:

# STEP 39 - Compact original vs updated T38 summary

cols = [
    "split",
    "original_series",
    "updated_series",
    "original_positive",
    "updated_positive",
    "original_targets",
    "updated_targets",
    "target_difference"
]

print("FINAL TARGET COMPARISON")
print(target_comparison[cols].to_string(index=False))

print("\nFINAL MEDIAN SKILL COMPARISON")
print(
    aggregate_comparison[
        ["split", "median_skill_original",
         "median_skill_updated"]
    ].to_string(index=False)
)

print("\nCHANGED TARGETS BY LOAD GROUP")
changed_groups = group_comparison[
    group_comparison["difference"] != 0
]

if changed_groups.empty:
    print("No changes by load group")
else:
    print(changed_groups.to_string(index=False))

print("\nFinal KPI comparison completed")


FINAL TARGET COMPARISON
     split  original_series  updated_series  original_positive  updated_positive  original_targets  updated_targets  target_difference
 test_2023               14              14                 14                14                 8                8                  0
 test_2025               11              11                  9                11                 5                6                  1
validation               14              14                 14                14                 7                7                  0

FINAL MEDIAN SKILL COMPARISON
     split  median_skill_original  median_skill_updated
 test_2023                  0.296                 0.296
 test_2025                  0.189                 0.201
validation                  0.307                 0.266

CHANGED TARGETS BY LOAD GROUP
    split      load_group  original_series  original_targets  updated_series  updated_targets  difference
test_2025 Plugs & Sockets                3  

## 4. Preliminary Audit Findings and Limitations

The revised T38 evaluation selected forecasting models across all 14 energy consumption series using the original selection procedure and updated GBM and Hybrid predictions.

The selected model distribution changed from 7 Chronos, 3 GBM, 3 Hybrid and 1 Random Forest to 10 Chronos, 2 corrected Hybrid, 1 Random Forest and 1 CatBoost.

All 14 selected series achieved positive skill against the daily naive benchmark on the 2023 test period. On the 2025 test period, positive skill increased from 9/11 to 11/11 series, while the number of series meeting their project-specific targets increased from 5/11 to 6/11.

### Methodological Limitations

**Partial leakage correction.** Random Forest and CatBoost were not retrained using strictly day-ahead features. However, according to the T32 feature-availability analysis, the NE zone has no indoor sensor measurements in the model-ready dataset. Therefore, the specific risk of same-hour indoor sensor leakage does not apply to the selected Random Forest model for NE HVAC.

Among the 14 updated selected models, NW Plugs & Sockets (CatBoost) is the only remaining winner potentially exposed to same-hour indoor sensor leakage.

Nevertheless, other forms of leakage have not been fully excluded, and the ranking of the original Random Forest and CatBoost candidates could change under a consistent strict day-ahead evaluation.

**Changing evaluation coverage.** The corrected models have fewer available predictions. Selection hours changed for 10 of the 14 series. Accordingly, original and updated CV(RMSE) and skill values are not always calculated on identical timestamps.

**Forecast availability.** The final 2025 evaluation includes 11 series, compared with 14 series in the 2023 test period.

**Interpretation.** The revised results represent a partial leakage-correction audit rather than definitive evidence that all model families are leakage-free. Differences in performance cannot be attributed exclusively to leakage correction.

Model selection was conducted on SEL. Test data were used only for subsequent evaluation, not to choose the winning models.

In [ ]:

# STEP 41 - Save audit comparison tables

from pathlib import Path

audit_dir = Path("/content/eval_T38_leakfree/audit_tables")
audit_dir.mkdir(parents=True, exist_ok=True)

outputs = {
    "T38_Model_Selection_Changes.csv": selection_changes,
    "T38_Selection_Hours_Audit.csv": selection_audit,
    "T38_Skill_Comparison.csv": skill_comparison,
    "T38_Target_Comparison.csv": target_comparison,
    "T38_Targets_By_Load_Group.csv": group_comparison,
    "T38_Aggregate_Metrics_Comparison.csv": aggregate_comparison
}

for filename, table in outputs.items():
    path = audit_dir / filename
    table.to_csv(path, index=False)
    print("Saved:", filename, "| Rows:", len(table))

assert all(
    (audit_dir / filename).exists()
    for filename in outputs
)

print("\nAudit tables export PASSED")


Saved: T38_Model_Selection_Changes.csv | Rows: 14
Saved: T38_Selection_Hours_Audit.csv | Rows: 14
Saved: T38_Skill_Comparison.csv | Rows: 14
Saved: T38_Target_Comparison.csv | Rows: 3
Saved: T38_Targets_By_Load_Group.csv | Rows: 12
Saved: T38_Aggregate_Metrics_Comparison.csv | Rows: 3

Audit tables export PASSED


In [ ]:

# STEP 42 - Locate T38 figures and T40 robustness scripts

from pathlib import Path

scripts_to_check = []

for path in sorted(project_dir.rglob("*.py")):
    name = path.name.lower()

    if (
        ("t38" in name and "figure" in name)
        or ("t40" in name and "robust" in name)
    ):
        scripts_to_check.append(path)

print("SCRIPTS FOUND:", len(scripts_to_check))

for path in scripts_to_check:
    print("\nFILE:", path.relative_to(project_dir))
    print("\nFIRST 35 LINES:")

    lines = path.read_text(
        encoding="utf-8",
        errors="replace"
    ).splitlines()

    for i, line in enumerate(lines[:35], start=1):
        print(f"{i:03d}: {line}")


SCRIPTS FOUND: 2

FILE: predictions_and_eval/TeamB_T38_figures.py

FIRST 35 LINES:
001: """
002: Figures for T38 (Team B). Reads Final_table from TeamB_T38_final_evaluation.xlsx.
003:   F13_best_model_vs_benchmark.png : CV(RMSE) of the chosen model vs the naive benchmark (validation, same hours)
004:   F14_skill_by_series.png         : skill score per series in validation, test 2023 and test 2025
005: Run: python TeamB_T38_figures.py eval_T38/TeamB_T38_final_evaluation.xlsx eval_T38
006: """
007: import sys, os
008: import numpy as np
009: import pandas as pd
010: import matplotlib
011: matplotlib.use("Agg")
012: import matplotlib.pyplot as plt
013: 
014: SURF, INK, INK2, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e1e0d9"
015: plt.rcParams.update({"figure.facecolor": SURF, "axes.facecolor": SURF, "savefig.facecolor": SURF,
016:                      "font.family": "DejaVu Sans", "text.color": INK, "axes.labelcolor": INK2,
017:                      "xtick.color": INK2, "ytick.color": INK2

In [ ]:

# STEP 43 - Verify updated T38 outputs for further analysis

required_outputs = [
    t38_output_dir / "TeamB_T38_final_evaluation.xlsx",
    t38_output_dir / "best_model_selection_final.csv",
    t38_output_dir / "best_model_predictions_final.csv",
    t36_output
]

print("UPDATED EVALUATION FILES")

for path in required_outputs:
    print(
        "FOUND" if path.exists() else "MISSING",
        path.name
    )

assert all(path.exists() for path in required_outputs)

print("\nUpdated T38 files verification PASSED")


UPDATED EVALUATION FILES
FOUND TeamB_T38_final_evaluation.xlsx
FOUND best_model_selection_final.csv
FOUND best_model_predictions_final.csv
FOUND all_model_predictions.csv

Updated T38 files verification PASSED


In [ ]:

# STEP 44 - Inspect figure and robustness script commands

for path in scripts_to_check:
    lines = path.read_text(
        encoding="utf-8",
        errors="replace"
    ).splitlines()

    print("\n" + "=" * 60)
    print("SCRIPT:", path.name)

    print("\nCOMMANDS AND INPUTS:")
    for i, line in enumerate(lines, start=1):
        if any(word in line.lower() for word in [
            "usage", "sys.argv", "argparse",
            "__main__", "read_csv", "read_excel",
            "savefig", "to_csv", "to_excel",
            "output", "mkdir"
        ]):
            print(f"{i:03d}: {line}")



SCRIPT: TeamB_T38_figures.py

COMMANDS AND INPUTS:
015: plt.rcParams.update({"figure.facecolor": SURF, "axes.facecolor": SURF, "savefig.facecolor": SURF,
021: xlsx, out = sys.argv[1], sys.argv[2]
022: f = pd.read_excel(xlsx, sheet_name="Final_table")
045: fig.savefig(os.path.join(out, "F13_best_model_vs_benchmark.png"), dpi=170); plt.close(fig)
071: fig.savefig(os.path.join(out, "F14_skill_by_series.png"), dpi=170); plt.close(fig)

SCRIPT: TeamB_T40_robustness_bias.py

COMMANDS AND INPUTS:
045: if __name__ == "__main__":
046:     pred_path, clean_dir, out = sys.argv[1:4]
052:     cl = pd.read_csv(os.path.join(clean_dir, "cedint_f1_hourly_clean.csv.gz"),
107:             d.round(3).to_excel(xw, sheet_name=name, index=False)
116:     plt.rcParams.update({"figure.facecolor": SURF, "axes.facecolor": SURF, "savefig.facecolor": SURF,
138:     fig.savefig(os.path.join(out, "F15_bias_recalibration.png"), dpi=170)


In [ ]:

# STEP 45 - Generate updated T38 figures

import subprocess
import sys
from pathlib import Path

figures_script = (
    project_dir /
    "predictions_and_eval/TeamB_T38_figures.py"
)

figures_dir = Path(
    "/content/eval_T38_leakfree/figures"
)

figures_dir.mkdir(parents=True, exist_ok=True)

command = [
    sys.executable,
    str(figures_script),
    str(evaluation_file),
    str(figures_dir)
]

result_figures = subprocess.run(
    command,
    capture_output=True,
    text=True
)

print("RETURN CODE:", result_figures.returncode)
print(result_figures.stdout[-5000:])

if result_figures.stderr:
    print("\nWARNINGS OR ERRORS:")
    print(result_figures.stderr[-3000:])

print("\nGenerated figures:")
figures = sorted(figures_dir.glob("*.png"))

for path in figures:
    print("-", path.name)

assert result_figures.returncode == 0
assert figures, "No figures generated"

print("\nT38 figures generation PASSED")


RETURN CODE: 0
figures written


Generated figures:
- F13_best_model_vs_benchmark.png
- F14_skill_by_series.png

T38 figures generation PASSED


In [ ]:

# STEP 46 - Run T40 robustness and bias analysis

robustness_script = (
    project_dir /
    "predictions_and_eval/TeamB_T40_robustness_bias.py"
)

robustness_dir = Path(
    "/content/eval_T38_leakfree/robustness"
)

robustness_dir.mkdir(parents=True, exist_ok=True)

command = [
    sys.executable,
    str(robustness_script),
    str(t38_output_dir / "best_model_predictions_final.csv"),
    str(project_dir),
    str(robustness_dir)
]

result_robustness = subprocess.run(
    command,
    capture_output=True,
    text=True
)

print("RETURN CODE:", result_robustness.returncode)
print("\nOUTPUT:")
print(result_robustness.stdout[-5000:])

if result_robustness.stderr:
    print("\nWARNINGS OR ERRORS:")
    print(result_robustness.stderr[-3000:])

print("\nGenerated files:")

robustness_files = sorted(
    p for p in robustness_dir.rglob("*") if p.is_file()
)

for path in robustness_files:
    print("-", path.name)

assert result_robustness.returncode == 0
assert robustness_files, "No robustness outputs generated"

print("\nT40 robustness analysis PASSED")


RETURN CODE: 0

OUTPUT:
zone       load_type      split  NMBE_before  NMBE_after  CV_before  CV_after  actual_vs_train_pct  share_flagged  NMBE_unflagged                                   verdict
  NE            HVAC  test_2023         59.4        45.8      691.6     691.0                134.1            0.1             6.8                      partly a level shift
  SE        Lighting  test_2023         26.4         1.2      167.2     197.0                272.9            0.2            -7.5 level shift: recalibration fixes the bias
  SE        Lighting  test_2025        -13.2       -20.8      257.4     272.3                302.3            0.3           -19.3               bias not explained by level
  SE        Lighting validation         30.5        -6.5      341.4     392.5                131.5            0.1           -93.1 level shift: recalibration fixes the bias
  SE Plugs & Sockets  test_2023         15.4        -0.7      127.3     136.2                -57.7            0.1   

## 5. Final Results and Conclusions

### 5.1 Objective

This notebook reconstructs the original T38 model-selection and evaluation workflow after replacing the original GBM and Hybrid predictions with the corrected outputs from Notebooks 02 and 03.

The evaluation covers 14 energy consumption series and retains the original project selection rules.

### 5.2 Updated Model Selection

Model selection was performed on the SEL period, independently of the 2023 and 2025 test periods.

| Model family | Original selection | Updated selection |
|---|---:|---:|
| Chronos | 7 | 10 |
| GBM | 3 | 0 |
| Hybrid | 3 | 2 |
| Random Forest | 1 | 1 |
| CatBoost | 0 | 1 |

**NE HVAC – Effective tie.** On the SEL period, Random Forest no2020 achieves a CV(RMSE) of 326.59%, compared with 326.62% for Chronos and 326.63% for the corrected Hybrid. Although Random Forest is technically selected as the winner, the differences are negligible (only 0.03–0.04 percentage points). Therefore, the result should be interpreted as an effective tie rather than evidence of a meaningful performance advantage. The series is also affected by meter anomalies, as identified in the T40 robustness analysis.


Six energy series changed their winning model family. One additional series retained the Hybrid family but switched to the corrected Hybrid version.

The corrected GBM was not selected as the best model for any of the 14 series.

### 5.3 Evaluation Results



| Metric | Original | Updated |
|---|---:|---:|
| Positive skill – Validation | 14/14 | 14/14 |
| Positive skill – Test 2023 | 14/14 | 14/14 |
| Positive skill – Test 2025 | 9/11 | 11/11 |
| **Positive skill – All periods** | **37/39** | **39/39** |
| Project targets met – Validation | 7/14 | 7/14 |
| Project targets met – Test 2023 | 8/14 | 8/14 |
| Project targets met – Test 2025 | 5/11 | 6/11 |
| Median skill – Validation | 0.307 | 0.266 |
| Median skill – Test 2023 | 0.296 | 0.296 |
| Median skill – Test 2025 | 0.189 | 0.201 |

The median validation skill decreased from 0.307 to 0.266 after the revised model selection.
This reduction is consistent with the removal of potentially inflated performance estimates from the original GBM and Hybrid predictions affected by data leakage. However, the change cannot be attributed exclusively to leakage correction because evaluation coverage and model selection also changed.

One of the two series that turned positive in the 2025 test period, NW Plugs & Sockets (CatBoost, skill score approximately 0.081), is also the only selected model still potentially exposed to same-hour indoor sensor leakage. Therefore, the improvement from 9/11 to 11/11 positive-skill series should be interpreted with this remaining limitation in mind.

The 2025 improvement in project-target achievement occurred in the Plugs & Sockets category, increasing from one to two successful series out of three.

These results describe the updated evaluation, but changes cannot be attributed solely to leakage correction because the selection and prediction coverage also changed.

### 5.4 Additional Evaluation

The original T38 figure-generation and T40 robustness scripts were executed using the updated model-selection outputs.

Both analyses completed successfully and produced updated figures and robustness reports.

### 5.5 Limitations

**Partial leakage correction.** The corrected GBM and Hybrid predictions were included in the revised evaluation. Random Forest and CatBoost were not retrained.

According to the T32 feature-availability analysis, the NE zone has no indoor sensor measurements in the model-ready dataset. Consequently, the selected Random Forest model for NE HVAC is not exposed to the specific risk of same-hour indoor sensor leakage.

Among the 14 selected models, NW Plugs & Sockets (CatBoost) is the only remaining winner potentially exposed to this type of leakage.

However, other forms of leakage have not been fully excluded. Moreover, the original Random Forest and CatBoost candidates were not consistently retrained using strictly day-ahead features, so their rankings could change under a fully standardized evaluation.

**Changing selection coverage.** The number of common SEL timestamps changed for 10 of the 14 series. Therefore, original and updated selection metrics are not always based on identical evaluation hours.

**Fixed-horizon interpretation.** The corrected GBM uses historical features designed for a fixed 24-hour forecasting horizon. Complete operational availability of every feature for a full next-day forecasting schedule has not been independently established.

**Test coverage.** The 2025 evaluation covers 11 series, compared with 14 series in the 2023 test period.

**Causal interpretation.** Observed performance differences cannot be attributed exclusively to leakage correction. Feature changes, training samples, prediction coverage and model selection may all contribute to the results.

### 5.6 Final Conclusion

The audit demonstrates that the original T38 model selection is sensitive to the corrected GBM and Hybrid inputs.

After the partial leakage correction, Chronos becomes the most frequently selected model family, winning 10 of the 14 energy series. The corrected Hybrid wins two series, while Random Forest and CatBoost each win one.

All selected models continue to outperform the daily naive benchmark on the 2023 test period. In 2025, positive skill increases from 9/11 to 11/11 series, and project targets met increase from 5/11 to 6/11.

However, this analysis should be interpreted as a partial leakage-correction audit, rather than a fully leakage-free evaluation across all candidate model families.

Further work should retrain Random Forest and CatBoost using strictly day-ahead features and validate all feature availability at the forecast issuance time.

### 5.7 Related Work

- `01_GBM_Timezone_Audit.ipynb` — UTC timestamp alignment.
- `02_GBM_Leakage_Correction.ipynb` — corrected GBM predictions for ten series.
- `03_Hybrid_Reevaluation.ipynb` — corrected Hybrid and FIT/FULL weight-estimation protocol.

The original project files were preserved. All revised evaluation outputs were written to separate directories.

In [ ]:

# STEP 48 - Final deliverables verification

from pathlib import Path

root = Path("/content/eval_T38_leakfree")

checks = {
    "Combined predictions": t36_output,
    "T38 evaluation": evaluation_file,
    "Updated model selection": selection_file,
    "Updated best predictions":
        t38_output_dir / "best_model_predictions_final.csv",
    "T38 figures folder": figures_dir,
    "T40 robustness folder": robustness_dir,
    "Audit tables folder": audit_dir
}

for label, path in checks.items():
    exists = path.exists()
    print(
        "PASSED" if exists else "MISSING",
        "|", label
    )
    assert exists, f"Missing: {path}"

assert len(list(figures_dir.glob("*.png"))) > 0
assert len(list(robustness_dir.glob("*"))) > 0
assert len(list(audit_dir.glob("*.csv"))) == 6

print("\nFINAL NOTEBOOK 04 VERIFICATION PASSED")


PASSED | Combined predictions
PASSED | T38 evaluation
PASSED | Updated model selection
PASSED | Updated best predictions
PASSED | T38 figures folder
PASSED | T40 robustness folder
PASSED | Audit tables folder

FINAL NOTEBOOK 04 VERIFICATION PASSED
